In [0]:
from pyspark.sql import functions as F

catalog = "vattenfall_dev"
schema = "raw"

landing_path = f"/Volumes/{catalog}/{schema}/landing/reference"
bronze_table = f"{catalog}.{schema}.bronze_asset_reference"

print("Landing path:", landing_path)
print("Bronze target table:", bronze_table)

display(dbutils.fs.ls(landing_path))


reference_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(landing_path)
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)


In [0]:
(
    reference_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

In [0]:
bronze_df = spark.table(bronze_table)

print("Rows in bronze after load:", bronze_df.count())
print("Columns in bronze:", bronze_df.columns)

display(bronze_df.limit(20))

In [0]:
for column_name in ["ingestion_ts", "source_file"]:
    if column_name in bronze_df.columns:
        print(f"Metadata column present: {column_name}")
    else:
        raise ValueError(f"Missing metadata column: {column_name}")